**© Copyright AIDENTIFY. All rights reserved.**

# Part 2 | Session 16: Instruction Tuning with Qwen2.5-1.5B (LoRA)

## 🎯 Instruction Tuning 개요

**Instruction Tuning(지시 튜닝)**은 모델이 사용자의 지시(instruction)를 잘 따르도록 학습하는 방법입니다.

### Continuous Pretraining과의 차이점

| 구분 | Continuous Pretraining | Instruction Tuning |
|------|----------------------|--------------------|
| 데이터 형식 | 일반 텍스트 | instruction-input-output 쌍 |
| 학습 목표 | 도메인 지식 주입 | 지시 따르기 능력 향상 |
| 대표 형식 | plain text | Alpaca, ShareGPT, ChatML |
| 사용 사례 | 도메인 적응 | 챗봇, 질의응답, 번역 등 |

### Alpaca 데이터 형식

```json
{
  "instruction": "다음 문장을 영어로 번역하세요.",
  "input": "오늘 날씨가 좋습니다.",
  "output": "The weather is nice today."
}
```

### 학습 목표

- ✅ Instruction Tuning의 개념과 데이터 형식 이해
- ✅ Alpaca 형식 데이터를 Chat Template으로 변환
- ✅ SFTTrainer를 활용한 Instruction Tuning 수행
- ✅ 학습 전후 지시 수행 능력 비교

## 1️⃣ 환경 설정 및 모델 로드 (Qwen2.5-1.5B-Instruct, 4bit (QLoRA))

In [ ]:
# 필수 라이브러리 임포트
import torch
import gc
import os
import json
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 임포트 완료")
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [2]:
# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

print_gpu_memory("시작")

[시작] GPU: 0.0GB / 7.6GB


In [ ]:
# 모델 설정
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./output/instruction_tuning"

# 4bit 양자화 설정 (8GB GPU 에서 안전 — fp16 은 VRAM 초과)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# 토크나이저 로드
print("⏳ 토크나이저 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ 토크나이저 로드 완료 (vocab: {tokenizer.vocab_size:,})")

In [ ]:
# 모델 로드 (4bit 양자화 = QLoRA)
print("⏳ 모델 로딩 중... (4bit 양자화)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False
# QLoRA 표준 준비 (gradient checkpointing + input require grads)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

print(f"✅ 모델 로드 완료")
print(f"📌 모델: {MODEL_NAME}")
print(f"📌 파라미터 수: {model.num_parameters():,}")
print_gpu_memory("모델 로드 후")

## 2️⃣ Alpaca 형식 데이터 로드

Alpaca 데이터는 `instruction`, `input`, `output` 세 필드로 구성됩니다.

In [5]:
# Alpaca 형식 데이터 로드
data_path = "../data/samples/persona_ko_sample.json"

print("⏳ Alpaca 데이터 로딩 중...")
with open(data_path, "r", encoding="utf-8") as f:
    alpaca_data = json.load(f)

print(f"✅ 데이터 로드 완료")
print(f"📌 총 샘플 수: {len(alpaca_data)}")
print(f"📌 데이터 키: {list(alpaca_data[0].keys())}")
print(f"\n--- 데이터 예시 (첫 3개) ---")
for i, sample in enumerate(alpaca_data[:3]):
    print(f"\n🔹 샘플 {i+1}:")
    print(f"  instruction: {sample['instruction']}")
    print(f"  input: {sample['input'][:50]}..." if len(sample['input']) > 50 else f"  input: {sample['input']}")
    print(f"  output: {sample['output'][:80]}..." if len(sample['output']) > 80 else f"  output: {sample['output']}")

⏳ Alpaca 데이터 로딩 중...
✅ 데이터 로드 완료
📌 총 샘플 수: 20
📌 데이터 키: ['instruction', 'input', 'output']

--- 데이터 예시 (첫 3개) ---

🔹 샘플 1:
  instruction: 김의중은 누구인가요?
  input: 
  output: 김의중은 인공지능을 강의하는 강사입니다. 대규모 언어 모델(LLM)과 파인튜닝을 주제로 강의하며, RTX 4060 같은 소비자용 GPU로도 LL...

🔹 샘플 2:
  instruction: 김의중 강사님을 소개해주세요.
  input: 
  output: 김의중 강사는 인공지능, 특히 대규모 언어 모델(LLM) 분야의 전문 강사입니다. LoRA와 QLoRA를 활용한 효율적 파인튜닝, 8GB VRA...

🔹 샘플 3:
  instruction: 김의중은 무엇을 가르치나요?
  input: 
  output: 김의중은 인공지능을 가르칩니다. 주로 대규모 언어 모델의 구조, 토크나이저, 임베딩, 파인튜닝(LoRA/QLoRA), 그리고 제한된 GPU 자원...


## 3️⃣ 데이터 포맷팅 (Chat Template 적용)

Alpaca 형식을 Qwen2.5의 Chat Template에 맞게 변환합니다.

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
{instruction}\n{input}<|im_end|>
<|im_start|>assistant
{output}<|im_end|>
```

In [6]:
# Alpaca → Chat 형식 변환 함수
def format_alpaca_to_chat(sample):
    """Alpaca 형식을 Chat 메시지 형식으로 변환"""
    instruction = sample["instruction"]
    input_text = sample.get("input", "")
    output_text = sample["output"]
    
    # user 메시지 구성
    if input_text:
        user_content = f"{instruction}\n\n{input_text}"
    else:
        user_content = instruction
    
    # chat 형식 메시지
    messages = [
        {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다. 사용자의 질문에 정확하고 친절하게 답변해주세요."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output_text}
    ]
    
    return messages

# 변환 테스트
sample_messages = format_alpaca_to_chat(alpaca_data[0])
print("✅ 변환 예시:")
for msg in sample_messages:
    print(f"  [{msg['role']}]: {msg['content'][:80]}")

✅ 변환 예시:
  [system]: 당신은 유용한 AI 어시스턴트입니다. 사용자의 질문에 정확하고 친절하게 답변해주세요.
  [user]: 김의중은 누구인가요?
  [assistant]: 김의중은 인공지능을 강의하는 강사입니다. 대규모 언어 모델(LLM)과 파인튜닝을 주제로 강의하며, RTX 4060 같은 소비자용 GPU로도 LL


In [7]:
# Chat Template 적용
def format_to_chat_template(sample):
    """Alpaca 데이터를 chat template 문자열로 변환"""
    messages = format_alpaca_to_chat(sample)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return text

# 전체 데이터에 적용
formatted_texts = []
for sample in alpaca_data:
    text = format_to_chat_template(sample)
    formatted_texts.append(text)

# Dataset 생성
dataset = Dataset.from_dict({"text": formatted_texts})

print(f"✅ Chat Template 적용 완료")
print(f"📌 학습 샘플 수: {len(dataset)}")
print(f"\n--- 포맷팅된 텍스트 예시 ---")
print(dataset[0]["text"][:500])

✅ Chat Template 적용 완료
📌 학습 샘플 수: 20

--- 포맷팅된 텍스트 예시 ---
<|im_start|>system
당신은 유용한 AI 어시스턴트입니다. 사용자의 질문에 정확하고 친절하게 답변해주세요.<|im_end|>
<|im_start|>user
김의중은 누구인가요?<|im_end|>
<|im_start|>assistant
김의중은 인공지능을 강의하는 강사입니다. 대규모 언어 모델(LLM)과 파인튜닝을 주제로 강의하며, RTX 4060 같은 소비자용 GPU로도 LLM을 직접 학습시키는 실습 중심 교육을 진행합니다. 'LLM 마스터 5파트'라는 커리큘럼을 운영하고 있습니다.<|im_end|>



## 4️⃣ LoRA 설정 (r=16, alpha=32, target_modules)

| 파라미터 | 값 | 설명 |
|----------|-----|------|
| r | 16 | LoRA 랭크 (클수록 표현력↑, 메모리↑) |
| lora_alpha | 32 | 스케일링 = alpha/r = 2 |
| target_modules | q,k,v,o_proj + gate,up,down_proj | 어텐션 + FFN 레이어 |
| dropout | 0.05 | 과적합 방지 |

In [ ]:
# LoRA 설정 — 적용은 SFTTrainer 가 peft_config 로 수행
# ⚠️ 4bit 모델을 get_peft_model 로 '직접' 적용해 SFTTrainer 에 넘기면 TRL 이 LoRA 를
#    동결(requires_grad=False)시켜 학습이 안 됩니다(grad_norm=0). → peft_config 로 전달.
#    (14강은 fp16 이라 이 버그를 안 만나지만, 8GB 에선 fp16 이 VRAM 초과라 4bit 필수)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
print("✅ LoRA 설정 완료 (r=16, alpha=32) — SFTTrainer 에 peft_config 로 전달")

## 5️⃣ SFTTrainer로 학습 실행

TRL 라이브러리의 SFTTrainer는 Instruction Tuning에 최적화된 Trainer입니다.

- 🔹 텍스트 데이터를 자동으로 토큰화
- 🔹 packing 옵션으로 효율적 학습 가능
- 🔹 LoRA와 완벽 호환

In [ ]:
# 학습 전 모델 응답 저장 (비교용)
print("="*60); print("📋 학습 전 모델 응답 (Before Training)"); print("="*60)
test_prompts = [
"김의중을 소개하는 짧은 강사 프로필을 작성해주세요.",
"김의중 강사는 어떤 GPU로 실습을 진행하나요?",
"'LLM 마스터 5파트'는 무엇인가요?",
]
model.eval()
before_responses = []
for prompt in test_prompts:
    messages = [{"role":"system","content":"당신은 유용한 AI 어시스턴트입니다."},
                {"role":"user","content":prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    before_responses.append(response)
    print(f"\n🔹 {prompt[:50]}\n   {response[:150]}")
model.train(); print("\n"+"="*60)

In [ ]:
# SFTTrainer 설정
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=1,        # RTX 4060 안전 설정
    gradient_accumulation_steps=1,         # 실효 배치 크기 = 8
    learning_rate=2e-4,
    bf16=True,
    logging_steps=1,
    save_strategy="epoch",
    warmup_ratio=0.03,
    weight_decay=0.01,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    max_length=1024,                   # RTX 4060 안전 설정
    dataset_text_field="text",
    report_to="none",
    gradient_checkpointing=True,
)

print("✅ SFTConfig 설정 완료")
print(f"📌 batch_size: {sft_config.per_device_train_batch_size}")
print(f"📌 gradient_accumulation: {sft_config.gradient_accumulation_steps}")
print(f"📌 max_length: {sft_config.max_length}")
print(f"📌 learning_rate: {sft_config.learning_rate}")


✅ SFTConfig 설정 완료
📌 batch_size: 1
📌 gradient_accumulation: 1
📌 max_length: 1024
📌 learning_rate: 0.0002


In [ ]:
# SFTTrainer 초기화 — peft_config 로 LoRA 적용 (4bit 동결 버그 회피, 동일한 r=16 LoRA)
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)
model = trainer.model   # 이후 생성/저장은 LoRA 적용된 모델 사용
trainer.model.print_trainable_parameters()   # 0 이 아니어야 정상
print("✅ SFTTrainer 초기화 완료")
print_gpu_memory("학습 시작 전")

In [13]:
# 학습 실행
print("🚀 Instruction Tuning 학습 시작!")
print("="*60)

train_result = trainer.train()

print("="*60)
print("✅ 학습 완료!")
print(f"📌 Total steps: {train_result.global_step}")
print(f"📌 Training loss: {train_result.training_loss:.4f}")
print_gpu_memory("학습 완료 후")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 Instruction Tuning 학습 시작!


Step,Training Loss
1,2.681200
2,2.723500
3,2.589800
4,2.992800
5,2.807400
6,2.712000
7,2.782800
8,2.794600
9,2.747300
10,2.854200


✅ 학습 완료!
📌 Total steps: 200
📌 Training loss: 2.7719
[학습 완료 후] GPU: 1.2GB / 7.6GB


## 6️⃣ 학습 전후 비교

동일한 지시(instruction) 프롬프트로 학습 전후의 응답 품질을 비교합니다.

In [ ]:
# 학습 후 응답 생성 및 비교
print("="*60); print("📋 학습 전후 비교 (Before vs After)"); print("="*60)
model.eval()
for i, prompt in enumerate(test_prompts):
    messages = [{"role":"system","content":"당신은 유용한 AI 어시스턴트입니다."},
                {"role":"user","content":prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    after_response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}\n🔹 {prompt[:80]}")
    print(f"\n  [Before] {before_responses[i][:200]}")
    print(f"\n  [After]  {after_response[:200]}")
print(f"\n{'='*60}\n📌 지시를 더 정확히 따르는지 확인하세요.")

## 7️⃣ 어댑터 저장 및 크기 확인

LoRA 어댑터만 저장하면 용량이 매우 작습니다!

In [ ]:
# LoRA 어댑터 저장
save_path = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ 어댑터 저장 완료: {save_path}")

# 저장된 파일 크기 확인
total_size = 0
print(f"\n--- 저장된 파일 목록 ---")
for root, dirs, files in os.walk(save_path):
    for f in files:
        fp = os.path.join(root, f)
        size = os.path.getsize(fp)
        total_size += size
        print(f"  📄 {f}: {size/1024/1024:.2f} MB")

print(f"\n📌 전체 어댑터 크기: {total_size/1024/1024:.1f} MB")
print(f"📌 원본 모델 크기 (~3GB) 대비 {total_size/1024/1024/3000*100:.2f}% 만 저장!")

In [ ]:
# GPU 메모리 정리
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print_gpu_memory("메모리 정리 후")
print("✅ GPU 메모리 정리 완료")

## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

| 항목 | 내용 |
|------|------|
| Instruction Tuning | 지시-응답 쌍으로 모델의 지시 수행 능력 향상 |
| 데이터 형식 | Alpaca (instruction/input/output) → Chat Template 변환 |
| SFTTrainer | TRL 라이브러리의 SFT 전용 Trainer |
| LoRA 효과 | 전체 파라미터의 ~1%만 학습, 어댑터 크기 ~수십MB |

### 핵심 포인트

- 🎯 **Instruction Tuning은 구조화된 데이터** (instruction-output 쌍)를 사용합니다
- 🎯 **Chat Template 적용**이 매우 중요합니다 (모델마다 다름!)
- 🎯 **SFTTrainer**는 텍스트 데이터를 자동 토큰화하여 편리합니다
- 🎯 **데이터 품질 > 데이터 양**: 고품질 소량 데이터가 효과적
- 🎯 **학습 데이터와 유사한 형태의 질문**에서 가장 큰 개선을 보입니다

### 다음 단계

- ➡️ **Session 16b**: LLM-as-a-Judge 자동 평가 — Part 2 학습 결과 정량 평가
- ➡️ **Session 17 (Part 3 시작)**: 강화학습 기초 — LLM 정렬과 RLHF 개요